In [1]:
from utils import *
import transformers 
import datasets

from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import JsonOutputParser,StrOutputParser
from tqdm import tqdm

c:\Users\rodri\OneDrive\Documentos\GitHub\AIIMS\out_experiments\EMAS\EMAS_REVIEW\bridging_memories\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
! ollama list

NAME                       ID              SIZE      MODIFIED     
deepseek-r1:8b             6995872bfe4c    5.2 GB    33 hours ago    
llama3.1:8b                46e0c10c039e    4.9 GB    33 hours ago    
nomic-embed-text:latest    0a109f422b47    274 MB    3 weeks ago     
phi4-mini:latest           78fad5d182a7    2.5 GB    3 weeks ago     
phi4-reasoning:latest      47e2630ccbcd    11 GB     3 weeks ago     
llama3.2:3b                a80c4f17acd5    2.0 GB    3 weeks ago     
gemma2:27b                 53261bc9c192    15 GB     7 weeks ago     
llama3.1:latest            46e0c10c039e    4.9 GB    2 months ago    
qwen2.5:32b                9f13ba1299af    19 GB     2 months ago    
phi3:medium                cf611a26b048    7.9 GB    2 months ago    
phi3:mini                  4f2222927938    2.2 GB    2 months ago    
phi:latest                 e2fd6321a5fe    1.6 GB    2 months ago    


In [3]:
train = datasets.Dataset.from_parquet("datasets/arc_challenge_train.parquet")
validation = datasets.Dataset.from_parquet("datasets/arc_challenge_validation.parquet")

Generating train split: 299 examples [00:00, 34760.71 examples/s]


# 1) Respondendo Perguntas

## 1.1) Resposta Estruturada

In [5]:
llama3 = ChatOllama(model='llama3.1:latest',temperature=0)
phi2 = ChatOllama(model='phi:latest',temperature=0)
gpt4mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
class Resposta(BaseModel):
    reasoning: str = Field(description="A brief description of the reasoning process that led to the choice of the alternative")
    alternative: str = Field(description="Only the letter of the alternative that is the correct answer to the question (A,B,C or D)")

parseador =  JsonOutputParser(pydantic_object=Resposta)

In [7]:
question = format_question(train[1])
print(question)

Which of the following statements best explains why magnets usually stick to a refrigerator door?
(A) The refrigerator door is smooth.
(B) The refrigerator door contains iron.
(C) The refrigerator door is a good conductor.
(D) The refrigerator door has electric wires in it.


In [16]:
prompt_template = PromptTemplate(
    template=""""
            Your goal is to answer the multiple-choice question below by applying scientific principles and using the provided context. If you do not know the answer, say you do not know: 
            {question}
            
            {format_instructions}
            """,
    input_variables=["question"],
    partial_variables={"format_instructions": parseador.get_format_instructions()}
)
model = phi2
cadeia = prompt_template | model | parseador

resposta = cadeia.invoke({"question": question})
print(resposta)

{'properties': {'reasoning': {'description': 'A brief description of the reasoning process that led to the choice of the alternative', 'title': 'Reasoning', 'type': 'string'}, 'alternative': {'description': 'Only the letter of the alternative that is the correct answer to the question (A, B, C or D)', 'title': 'Alternative', 'type': 'string'}}, 'required': ['reasoning', 'alternative']}


## 1.2) Resposta Não Estruturada

In [12]:
prompt_template = PromptTemplate(
    template=""""
            Your goal is to answer the multiple-choice question below by applying scientific principles and using the provided context. If you do not know the answer, say you do not know: 
            {question}
            """,
    input_variables=["question"])

model = phi2
cadeia = prompt_template | model | StrOutputParser()

resposta = cadeia.invoke({"question": question})
print(resposta)

 (B) The refrigerator door contains iron. Magnets are attracted to ferromagnetic materials, such as iron, nickel, and cobalt. These materials have domains that align with an external magnetic field, creating a strong attraction between the magnet and the material. In this case, the refrigerator door is made of metal, which can contain iron or other ferromagnetic elements.



# 2) Fazendo o proccesso completo

In [18]:
class Resposta(BaseModel):
    reasoning: str = Field(description="A brief description of the reasoning process that led to the choice of the alternative")
    alternative: str = Field(description="Only the letter of the alternative that is the correct answer to the question (A,B,C or D)")

parseador =  JsonOutputParser(pydantic_object=Resposta)

In [20]:
formatted_template = PromptTemplate(
    template="""Instruction: You are an expert science tutor. Your goal is to answer the target multiple-choice question below.

            Apply relevant scientific principles - fundamental definitions and laws from your knowledge.

            Use scientific principles to ground your facts and guide your logic.
            ### TARGET QUESTION:
            {question}

            {format_instructions}
            """,
    input_variables=["question"],
    partial_variables={"format_instructions": parseador.get_format_instructions()},
)

simple_template = PromptTemplate(
    template="""Instruction: You are an expert science tutor. Your goal is to answer the target multiple-choice question below.

            Apply relevant scientific principles - fundamental definitions and laws from your knowledge.

            Use scientific principles to ground your facts and guide your logic.

            Structure your response strictly as:
            1. **Reasoning:** Explain the step-by-step logic to reach the correct answer.
            2. **Answer:** State only the correct option letter (A, B, C, or D).

            ### TARGET QUESTION:
            {question}
            """,
    input_variables=["question"],
)

In [37]:
def answer_question(model, question: str, structure_output: bool = False):
    if structure_output:
        cadeia = formatted_template | model | parseador
    else:
        cadeia = simple_template | model | StrOutputParser()
    resposta = cadeia.invoke({"question": question})
    return resposta


def answer_dataset(model, dataset, structure_output: bool = False) -> dict:
    output = {}
    acertos = []
    loop = tqdm(range(len(dataset)))
    for i in loop:
        question = format_question(dataset[i])
        full_answer = answer_question(model, question, structure_output)

        correct_answer_alt = get_correct_answer(dataset[i])
        correct_answer_text = get_correct_answer_text(dataset[i])
        predicted_alt = detect_answer(
            full_answer,
            valid_labels=["A", "B", "C", "D"],
            judge=gpt4mini,
            structure_output=structure_output,
            question=question,  # ← PASS THE CURRENT QUESTION
        )
        acerto = evaluate_answer(predicted_alt, correct=correct_answer_alt)
        acertos.append(acerto)
        output[i] = {
            "question": question,
            "answer": full_answer,
            "correct_answer_alt": correct_answer_alt,
            "predicted_alt": predicted_alt,
            "correct_answer_text": correct_answer_text,
            "flag_acerto": acerto,
        }

        acc = sum(acertos) / len(acertos)
        loop.set_postfix(acc=f"{acc:.1%}")
    return output


def detect_answer(
    resposta: str,
    valid_labels=["A", "B", "C", "D"],
    judge=None,
    structure_output: bool = False,
    debug: bool = False,
    question: str = None,
) -> str:
    # Se for estruturado, então busca no dicionário; Se não for estruturado então devemos usar regex para extrair a letra da resposta
    if structure_output:
        if "alternative" in resposta:
            return resposta["alternative"]
        else:  # Adicionar logs posteriormente para entender o motivo de não encontrar a resposta
            return "N/A"
    else:
        extracted_answer = extract_answer(
            judge, resposta, valid_labels=valid_labels, question=question, debug=debug
        )
        return extracted_answer


def evaluate_answer(predicted: str, correct: str) -> bool:
    correct_answer = correct.strip().upper()
    treated_guess = predicted.strip().upper()
    if correct_answer == treated_guess:
        return True
    else:
        return False


In [38]:
resposta_estruturada = answer_question(llama3, question, structure_output=True)
print(resposta_estruturada)

{'alternative': 'B', 'reasoning': 'Magnets stick to refrigerator doors because they contain iron, which is a ferromagnetic material that can be magnetized and thus attracted by magnets.'}


In [23]:
resposta = answer_question(llama3, question, structure_output=False)
print(resposta)

**Reasoning:**

1. First, let's recall the fundamental definition of magnetism: Magnetism is a physical phenomenon resulting from the interaction between magnetic fields and other materials that can be magnetized.
2. Next, we need to identify the type of material that magnets are attracted to. Magnets are attracted to ferromagnetic materials, which include iron, nickel, and cobalt.
3. Now, let's analyze each option: 
   - Option A is incorrect because smoothness has no relation to magnetism.
   - Option B suggests that the refrigerator door contains iron, which aligns with our understanding of magnetism.
   - Option C is incorrect because being a good conductor (i.e., allowing electricity to flow through) does not affect magnetic attraction.
   - Option D is also incorrect as electric wires do not contribute to magnetic attraction in this context.
4. Based on the principles of magnetism, we can conclude that magnets stick to a refrigerator door primarily due to the presence of iron.

*

In [ ]:
# resposta_estruturada = answer_question(phi2, question, structure_output=True)
# print(resposta_estruturada)

In [24]:
resposta = answer_question(phi2, question, structure_output=False)
print(resposta)

 
1. Reasoning: To answer this question, we need to understand the properties of magnets and how they interact with different surfaces.
2. Answer: (B) The refrigerator door contains iron.

Explanation: Magnets usually stick to a refrigerator door because the door is made of iron, which is a magnetic material. When a magnet comes into contact with an iron surface, it creates a magnetic field that attracts the magnet to the door. This is due to the alignment of the magnetic domains within the iron atoms in the door's surface. The smoothness or conductivity of the door does not play a significant role in this phenomenon.



# Avaliando Modelos

In [27]:
phi2 = ChatOllama(model='phi:latest',temperature=0)
llama3 = ChatOllama(model='llama3.1:latest',temperature=0)
gpt4mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [30]:
selected = validation.select(range(100))

In [39]:
# phi2_structured = answer_dataset(phi2,selected, structure_output=True, debug=False, description="Phi2 estruturado")
phi2_non_structured = answer_dataset(phi2,selected, structure_output=False)

100%|██████████| 100/100 [02:11<00:00,  1.32s/it, acc=71.0%]


In [41]:
print("Llamma 3 Não Estruturado")
ollama_non_structured = answer_dataset(llama3,selected, structure_output=False)

print("GPT-4.0 Mini Não Estruturado")
gpt40_non_structured = answer_dataset(gpt4mini,selected, structure_output=False)

Llamma 3 Não Estruturado


100%|██████████| 100/100 [09:58<00:00,  5.98s/it, acc=75.0%]


GPT-4.0 Mini Não Estruturado


100%|██████████| 100/100 [07:06<00:00,  4.27s/it, acc=88.0%]


In [42]:
print("Llamma 3 Estruturado")
ollama_structured = answer_dataset(llama3,selected, structure_output=True)

print("GPT-4.0 Mini Estruturado")
gpt40_structured = answer_dataset(gpt4mini,selected, structure_output=True)

Llamma 3 Estruturado


100%|██████████| 100/100 [03:58<00:00,  2.39s/it, acc=87.0%]


GPT-4.0 Mini Estruturado


100%|██████████| 100/100 [03:13<00:00,  1.93s/it, acc=92.0%]
